In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    sum,
    avg,
    when,
    round,
    lower,
    trim,
    regexp_replace
)

spark = (
    SparkSession.builder
    .appName("Gaming Analytics - Gold Layer")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.local.dir", "D:/spark_temp")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

In [17]:
print("\n========== LOAD SILVER DATA ==========\n")


games = spark.read.parquet(
    "../data/silver/games_clean"
)
reviews = spark.read.parquet(
    "../data/silver/reviews_clean"
)
rawg = spark.read.parquet(
    "../data/silver/rawg_games_clean"
)

print("Games:")
games.printSchema()
print("\nReviews:")
reviews.printSchema()
print("\nRAWG:")
rawg.printSchema()


========== LOAD SILVER DATA ==========

Games:
root
 |-- appid: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- english: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- platforms: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- steamspy_tags: string (nullable = true)
 |-- achievements: string (nullable = true)
 |-- positive_ratings: integer (nullable = true)
 |-- negative_ratings: integer (nullable = true)
 |-- average_playtime: integer (nullable = true)
 |-- median_playtime: integer (nullable = true)
 |-- owners: string (nullable = true)
 |-- price: double (nullable = true)


Reviews:
root
 |-- app_id: integer (nullable = true)
 |-- review_text: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_votes: integer (nullable = tr

In [18]:
print("\n========== AGGREGATE REVIEWS ==========\n")

review_summary = (
    reviews
    .groupBy("app_id")
    .agg(
        count("*")
        .alias("total_reviews"),
        sum(
            when(
                col("review_score") == 1,
                1
            )
            .otherwise(0)
        )
        .alias("positive_reviews"),
        sum(
            when(
                col("review_score") == 0,
                1
            )
            .otherwise(0)
        )
        .alias("negative_reviews"),
        avg(
            col("review_score")
        )
        .alias("average_review_score")
    )
)


========== AGGREGATE REVIEWS ==========



In [19]:
review_summary = review_summary.withColumn(
    "recommendation_rate",
    round(
        col("positive_reviews") /
        col("total_reviews"),
        3
    )
)

In [20]:
games = games.withColumnRenamed(
    "appid",
    "app_id"
)

In [21]:
games = games.withColumn(
    "game_name_clean",
    lower(
        trim(
            regexp_replace(
                col("name"),
                "[^a-zA-Z0-9 ]",
                ""
            )
        )
    )
)

In [22]:
print("\n========== JOIN STEAM DATA ==========\n")


steam_analysis = games.join(
    review_summary,
    "app_id",
    "left"
)


========== JOIN STEAM DATA ==========



In [23]:
print("\n========== JOIN RAWG DATA ==========\n")

gold_data = steam_analysis.join(
    rawg.select(
        "game_name_clean",
        col("rating")
        .alias("rawg_rating"),
        col("metacritic")
        .alias("rawg_metacritic")
    ),
    "game_name_clean",
    "left"
)


========== JOIN RAWG DATA ==========



In [24]:
# %%
print("\n========== HANDLE MISSING VALUES ==========\n")

gold_data = gold_data.fillna(
    {
        "total_reviews":0,
        "positive_reviews":0,
        "negative_reviews":0,
        "recommendation_rate":0,
        "average_review_score":0,
        "positive_ratings":0,
        "negative_ratings":0
    }
)


# Convert missing RAWG values only
gold_data = gold_data.withColumn(
    "rawg_rating",
    when(
        col("rawg_rating") == 0,
        None
    )
    .otherwise(col("rawg_rating"))
)


gold_data = gold_data.withColumn(
    "rawg_metacritic",
    when(
        col("rawg_metacritic") == 0,
        None
    )
    .otherwise(col("rawg_metacritic"))
)


print("Missing value handling completed")


========== HANDLE MISSING VALUES ==========

Missing value handling completed


In [25]:
gold_data = gold_data.select(
    "app_id",
    "name",
    "developer",
    "publisher",
    "genres",
    "price",
    "average_playtime",
    "positive_ratings",
    "negative_ratings",
    "total_reviews",
    "positive_reviews",
    "negative_reviews",
    "recommendation_rate",
    "average_review_score",
    "rawg_rating",
    "rawg_metacritic"
)

In [26]:
print("\n========== GOLD DATASET ==========\n")

gold_data.printSchema()

print(
    "Gold rows:",
    gold_data.count()
)
gold_data.show(
    10,
    truncate=False
)


========== GOLD DATASET ==========

root
 |-- app_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- price: double (nullable = true)
 |-- average_playtime: integer (nullable = true)
 |-- positive_ratings: integer (nullable = false)
 |-- negative_ratings: integer (nullable = false)
 |-- total_reviews: long (nullable = false)
 |-- positive_reviews: long (nullable = false)
 |-- negative_reviews: long (nullable = false)
 |-- recommendation_rate: double (nullable = false)
 |-- average_review_score: double (nullable = false)
 |-- rawg_rating: double (nullable = true)
 |-- rawg_metacritic: integer (nullable = true)

Gold rows: 27075
+------+------------------------------+---------------------+---------------------+----------------------------+-----+----------------+----------------+----------------+-------------+----------------+----------------+--------

In [27]:
print("\n========== SAVE GOLD DATA ==========\n")


gold_data.repartition(1).write \
    .mode("overwrite") \
    .parquet(
        "../data/gold/game_analysis"
    )


print("Gold layer saved successfully")


========== SAVE GOLD DATA ==========

Gold layer saved successfully


In [28]:
print("\n========== LOAD GOLD DATA CHECK ==========\n")

gold_check = spark.read.parquet(
    "../data/gold/game_analysis"
)

gold_check.printSchema()

print("Rows:")
print(gold_check.count())

print("\n========== GOLD SCHEMA ==========\n")

gold_check.printSchema()

from pyspark.sql.functions import count, when, col


print("\n========== GOLD NULL CHECK ==========\n")


gold_check.select([
    count(
        when(
            col(c).isNull(),
            c
        )
    ).alias(c)
    for c in gold_check.columns
]).show()

print("\n========== DUPLICATE GAME CHECK ==========\n")


duplicates = (
    gold_check
    .groupBy("app_id")
    .count()
    .filter(
        col("count") > 1
    )
)


print(
    "Duplicate games:",
    duplicates.count()
)

print("\n========== RECOMMENDATION RATE CHECK ==========\n")


gold_check.select(
    "recommendation_rate"
).describe().show()

print("\n========== RAWG MATCH CHECK ==========\n")


gold_check.filter(
    col("rawg_rating") > 0
).count()
gold_check.filter(
    col("rawg_rating") == 0
).count()

gold_check.select(
    "name",
    "genres",
    "price",
    "average_playtime",
    "recommendation_rate",
    "rawg_rating"
).show(
    20,
    truncate=False
)


========== LOAD GOLD DATA CHECK ==========

root
 |-- app_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- price: double (nullable = true)
 |-- average_playtime: integer (nullable = true)
 |-- positive_ratings: integer (nullable = true)
 |-- negative_ratings: integer (nullable = true)
 |-- total_reviews: long (nullable = true)
 |-- positive_reviews: long (nullable = true)
 |-- negative_reviews: long (nullable = true)
 |-- recommendation_rate: double (nullable = true)
 |-- average_review_score: double (nullable = true)
 |-- rawg_rating: double (nullable = true)
 |-- rawg_metacritic: integer (nullable = true)

Rows:
27075

========== GOLD SCHEMA ==========

root
 |-- app_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genres: string (nullable =